!!! Note: CLADE is commiting data leakage, because only the 149,361 / 160,000 variants screened in the dataset are considered in the simulation and can be chosen by the algorithm. the 10,639 missing variants should be considered to have 0 fitness. This data leakage improves CLADE's percieved performance compared to reality.

CLADE 2.0 paper uses:
NUM_first = 96
NUM_hierarchy = 96
NUM_train = 384 (4 * 96)
M = 96 (how many variants are screened during exploitation of the trained ML model)

total = NUM_TRAIN + M = (NUM_first + rest of NUM_train) + M = (1/5 + 3/5) + 1/5
50 = (10 + 3*10) + 10       10 0 10 10 --batch_size 1 --num_first_round 10 --hierarchy_batch 10 --num_batch 40 (then M=10)
100 = (20 + 3*20) + 20      10 0 10 10 --batch_size 1 --num_first_round 20 --hierarchy_batch 20 --num_batch 80 (then M=20)
150 = (30 + 3*30) + 30      10 0 10 10 --batch_size 1 --num_first_round 30 --hierarchy_batch 30 --num_batch 120 (then M=30)
190 = (38 + 3*38) + 38      10 0 10 10 --batch_size 1 --num_first_round 38 --hierarchy_batch 38 --num_batch 152 (then M=38)

First run:
```
python CLADE2.py 10 0 10 10 --dataset <DATASET_NAME> --batch_size 24 --num_first_round 24 --hierarchy_batch 24 --num_batch 1
```
or
```
python CLADE2.py 10 0 10 10 --dataset <DATASET_NAME> --batch_size 1 --num_first_round 1 --hierarchy_batch 1 --num_batch 24
# 20250423-111505 -> max = 5.33546016514 (0.6089341564199108)

python CLADE2.py 10 0 10 10 --dataset <DATASET_NAME> --batch_size 1 --num_first_round 6 --hierarchy_batch 1 --num_batch 24
# 20250423-111256 -> max = 4.82340475447 (0.550493455920668)
```

In [ ]:
import pandas as pd
import numpy as np

In [12]:
DATASET_NAME = "GB1" # GB1, PhoQ

# Define the file path
SAMPLING_RUN_ID = "20250423-111256" # "20250423-111505", "20250423-111256"
MLDE_RUN_ID = "20250423-111306" # "20250423-111510", "20250423-111306"

file_path = "/Users/soldatmat/Documents/boes/CLADE-2.0/"+SAMPLING_RUN_ID+"/"+MLDE_RUN_ID+"/PredictedFitness.csv"
data = pd.read_csv(file_path)
data

,AACombo,PredictedFitness,InTrainingData?
0,LHGA,0.326940,YES
1,LHSA,0.314446,NO
2,LHAA,0.314412,NO
3,LHTA,0.314292,NO
4,LHPA,0.314250,NO
...,...,...,...
149356,EDCM,0.008085,NO
149357,HDCF,0.008072,NO
149358,MDCF,0.008069,NO
149359,HDCM,0.008061,NO


In [13]:
N_SCREEN = 56

selection = []
i = 0
while len(selection) < N_SCREEN:
    if data["InTrainingData?"][i] == "NO":
        selection.append(i)
    i += 1

assert len(selection) == N_SCREEN
selected_variants = data.iloc[selection]

train_variants = data[data["InTrainingData?"] == "YES"]

screened_variants = pd.concat([train_variants, selected_variants])

In [14]:
data_path = "/Users/soldatmat/Documents/boes/CLADE-2.0/"+DATASET_NAME+"/"+DATASET_NAME+".xlsx"

groundtruth = pd.read_excel(data_path)

# Normalize fitness scores
groundtruth['NormalizedFitness'] = groundtruth['Fitness'].values / groundtruth['Fitness'].values.max()

screened_variants.rename(columns={'AACombo': 'Variants'}, inplace=True)
screened_variants = pd.merge(groundtruth, screened_variants, on='Variants', how='inner')
screened_variants = screened_variants.sort_values(by='Fitness', ascending=False)
screened_variants

/opt/miniconda3/envs/CLADE/lib/python3.11/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Unknown extension is not supported and will be removed
  for idx, row in parser.parse():


,Variants,HD,Count input,Count selected,Fitness,NormalizedFitness,PredictedFitness,InTrainingData?
44,LHCA,4,356,6265,4.823405,0.550493,0.314232,NO
40,LHAA,4,877,13104,4.095311,0.467396,0.314412,NO
0,LHGV,2,613,7508,3.356962,0.383129,0.307486,NO
12,LHGA,3,942,11379,3.310822,0.377863,0.326940,YES
46,LHCG,4,136,1609,3.242648,0.370082,0.313551,NO
...,...,...,...,...,...,...,...,...
50,LHEA,4,242,3,0.003398,0.000388,0.314211,NO
49,LHDG,4,172,2,0.003187,0.000364,0.313564,NO
55,LHKG,4,87,1,0.003150,0.000360,0.313564,NO
48,LHDC,4,129,1,0.002125,0.000242,0.307288,NO


In [17]:
max_fitness = max(screened_variants['NormalizedFitness'])
max_fitness

0.550493455920668

In [ ]:
output_path = "/path/to/save/screened_variants.csv"
screened_variants.to_csv(output_path, index=False)

np.save("/path/to/save/max_fitness.npy", max_fitness)